# 29b · Redraw Figures 2 and 7 and Supplementary Figure 11 from the saved notebook 29 results (review rounds 3 and 4)

Reads only `04_results/revisions_c2/round2_policies_c2.json` (written by notebook 29) and the cycle 1 aggregate files; no predictions, no new analysis. Changes against the figures notebook 29 drew: Figure 2b subtitle on one line; Figure 7 step 3 and panel d exclude DDR (too few pilot groups) and label intervals as conditional on the pilot draws; Supplementary Figure 11a, b use the same six non-DDR rosters at every pilot size. Superseded figures are renamed `SUPERSEDED_`. CPU, under a minute.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import os, json, datetime
import numpy as np, pandas as pd
DATA = "/content/drive/MyDrive/stochastic_orthogonality/data"
BASE = f"{DATA}/dr_ensemble_v2"; DERM = f"{DATA}/derm"; OUT = f"{BASE}/05_figures/manuscript_final"
REC = json.load(open(f"{BASE}/04_results/revisions_c2/round2_policies_c2.json"))
RESA, SUMA, RESB = REC["A_policies"], REC["A_summary"], REC["B_max_vs_mean_rank"]
POL = ["better", "blind", "pick_auc", "pick_vol", "pair_max", "pair_mean", "adapt_delong_max", "adapt_auc_max", "adapt_auc_mean", "adapt_vol_max", "adapt_vol_mean"]
print("loaded", REC["written_utc"])

Mounted at /content/drive
loaded 2026-10-31T01:44:51.631021+00:00


In [2]:
# Shared style
import shutil
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': '#333333', 'axes.linewidth': 0.8, 'xtick.color': '#333333', 'ytick.color': '#333333',
                     'axes.labelcolor': '#222222', 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.labelsize': 8.5, 'legend.frameon': False})
BLUE, SKY, ORANGE, VERM, PINK, GREY, INK, GREEN = '#0072B2', '#56B4E9', '#E69F00', '#D55E00', '#CC79A7', '#8C8C8C', '#222222', '#009E73'
DOMCOL = {'retina': BLUE, 'dermatology': VERM}
SITES = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']
SNAME = {'EyePACS': 'EyePACS (US)', 'DDR': 'DDR (China)', 'BRSET': 'BRSET (Brazil)', 'HIBA': 'HIBA (Argentina)', 'PAD': 'PAD-UFES-20 (Brazil)'}
SDOM = {'EyePACS': 'retina', 'DDR': 'retina', 'BRSET': 'retina', 'HIBA': 'dermatology', 'PAD': 'dermatology'}
MARK = {'EyePACS': 'o', 'DDR': 's', 'BRSET': 'P', 'HIBA': '^', 'PAD': 'D'}
LS = {'EyePACS': '-', 'DDR': '--', 'BRSET': ':', 'HIBA': '-', 'PAD': '--'}
CNAME = {'moderate': 'Moderate NPDR', 'severe NPDR': 'Severe NPDR', 'PDR': 'PDR', 'BCC': 'BCC', 'SCC': 'SCC', 'melanoma': 'Melanoma'}
# relationship of two systems: one display name and one colour, used in Figures 3, 5 and 6
RELMAP = {'different design, different data': 'Different design, different data', 'different design, same data': 'Different design, same data',
          'different design': 'Different design, same data', 'same checkpoint, different data': 'Same design, different data',
          'variant sibling': 'Variant sibling (same design and data)', 'seed sibling': 'Seed sibling (repeat run)'}
RELS = ['Different design, different data', 'Different design, same data', 'Same design, different data', 'Variant sibling (same design and data)', 'Seed sibling (repeat run)']
RELCOL = dict(zip(RELS, [BLUE, SKY, ORANGE, PINK, VERM]))
def ci_txt(m, lo, hi, d=2, pct=False):
    u = '%' if pct else ''
    return f'{m:.{d}f}{u} ({lo:.{d}f} to {hi:.{d}f})'
def panel(ax, letter, text, sub=None):
    ax.set_title(f'{letter}  {text}', loc='left', fontsize=9.5, weight='bold', pad=14 if sub else 6)
    if sub: ax.text(0, 1.015, sub, transform=ax.transAxes, fontsize=8, color='#444444', va='bottom')
def title(fig, t): fig.suptitle(t, x=0.01, ha='left', fontsize=11, weight='bold')
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d%H%M')
def save(fig, name):
    for ext in ('png', 'pdf'):
        p = f'{OUT}/{name}.{ext}'
        if os.path.exists(p):
            dest = f'{OUT}/SUPERSEDED_{name}.{ext}'
            if os.path.exists(dest): dest = f'{OUT}/SUPERSEDED_{name}_{STAMP}.{ext}'
            shutil.move(p, dest)
    fig.savefig(f'{OUT}/{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)
DR = json.load(open(f'{DERM}/derm_results_v1.json'))
RESULTS = f'{BASE}/04_results'
REV = f'{RESULTS}/revisions_c1'
REG = json.load(open(f'{REV}/regret_c1.json'))['results']; ADA = json.load(open(f'{REV}/adaptive_rule_c1.json'))['results']
DEP = json.load(open(f'{REV}/deployment_c1.json')); POOL = json.load(open(f'{REV}/pooled_vs_pairing_c1.json'))
PAIRS = pd.read_csv(f'{REV}/regret_c1_pairs.csv'); BRR = json.load(open(f'{RESULTS}/brset_results_v6.json'))
from matplotlib.patches import FancyBboxPatch
def box(ax, x, y, w, h, head, body, ec, fc, hs=10, bs=8.3, lw=1.4, ital=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.004,rounding_size=0.01', ec=ec, fc=fc, lw=lw, transform=ax.transAxes))
    if head: ax.text(x + w / 2, y + h - 0.016, head, ha='center', va='top', fontsize=hs, weight='bold', color=INK, transform=ax.transAxes)
    if body: ax.text(x + w / 2, (y + h - 0.05) if head else (y + h / 2), body, ha='center', va='top' if head else 'center', fontsize=bs, color=INK,
                     linespacing=1.35, style='italic' if ital else 'normal', transform=ax.transAxes)
def arrow(ax, x0, y0, x1, y1, col='#666666'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), xycoords='axes fraction', arrowprops=dict(arrowstyle='-|>', color=col, lw=1.3, shrinkA=0, shrinkB=0))
def gtag(ax, x, y, t='confirmatory, dated plan'):
    ax.text(x, y, t, ha='center', va='center', fontsize=7.4, color='white', weight='bold', transform=ax.transAxes, bbox=dict(boxstyle='round,pad=0.3', fc=GREEN, ec='none'))
def blank(ax): ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
LBL = {'EyePACS_severe': 'EyePACS, severe DR', 'DDR_severe': 'DDR, severe DR', 'DDR_pdr': 'DDR, PDR', 'BRSET_severe': 'BRSET, severe DR',
       'BRSET_mixed_severe': 'BRSET, mixed sources, severe DR', 'HIBA_mel': 'HIBA, melanoma', 'HIBA_mal': 'HIBA, any skin cancer',
       'PAD_mel': 'PAD-UFES-20, melanoma', 'PAD_mal': 'PAD-UFES-20, any skin cancer'}
LBL_DEP = {**LBL, 'DDR_pdr': 'DDR, PDR'}
KC = {'EyePACS_severe': '#0072B2', 'DDR_severe': '#56B4E9', 'DDR_pdr': '#009E73', 'BRSET_severe': '#003A70', 'BRSET_mixed_severe': '#8C8C8C',
      'HIBA_mel': '#D55E00', 'HIBA_mal': '#E69F00', 'PAD_mel': '#CC79A7', 'PAD_mal': '#8B4513'}
KM = lambda k: '^' if k.split('_')[0] in ('HIBA', 'PAD') else 'o'
def p0(x): return f'{int(np.floor(100 * x + 0.5))}%'   # percentages rounded half up, as in the text

FD = json.load(open(f'{OUT}/figures_c1_data.json')); DET = FD['sibling_detector']; SITES5 = ['EyePACS', 'DDR', 'BRSET', 'HIBA', 'PAD']


In [3]:
# Figure 2: benefit at the same referral volume; (a) against the average member, (b) against the better member on a count scale, (c) skin strongest partner
H1 = DR['H1_benefit']; b2 = BRR['B2_benefit']['a_eyepacs_all_110_pairings']['sev']
def pct_(r, k='relative_reduction_matched_volume'): return 100 * r[k], 100 * r['ci95'][0], 100 * r['ci95'][1]
ROWSA = [('EyePACS (US)\nPDR (secondary)', (40, 30, 51), 'retina', False), ('DDR (China)\nProliferative DR', (27, 19, 40), 'retina', True),
         ('BRSET (Brazil)\nSevere DR (grade 3 or 4)', pct_(b2, 'relative_reduction'), 'retina', True), ('HIBA (Argentina)\nAny skin cancer', pct_(H1['HIBA_f']['mal']), 'dermatology', True),
         ('HIBA (Argentina)\nMelanoma', pct_(H1['HIBA_f']['mel']), 'dermatology', True), ('PAD-UFES-20 (Brazil)\nAny skin cancer', pct_(H1['PAD_f']['mal']), 'dermatology', True)]
NEY, NDDR = 15794, 12422
ROWSB = [('EyePACS (US)\nSevere DR, pre-specified primary', tuple(v / NEY * 1e4 for v in (4.74, 2.31, 6.15)), 'retina', True),
         ('DDR (China)\nProliferative DR', tuple(v / NDDR * 1e4 for v in (3.796, 1.547, 6.39)), 'retina', False),
         ('BRSET (Brazil)\nSevere DR', (-1.8, -3.9, -0.3), 'retina', False),
         ('HIBA (Argentina)\nAny skin cancer', (26.4, 2.5, 41.6), 'dermatology', False), ('HIBA (Argentina)\nMelanoma', (-7.0, -24.4, 6.4), 'dermatology', False),
         ('PAD-UFES-20 (Brazil)\nAny skin cancer', (21.8, -7.1, 40.2), 'dermatology', False), ('PAD-UFES-20 (Brazil)\nMelanoma', (-6.5, -17.9, -1.8), 'dermatology', False)]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6), gridspec_kw={'width_ratios': [1, 1.05, 0.9]})
def forest(ax, rows, unit, xl, fmtv):
    for i, (lab, (m, lo, hi), dom, pre) in enumerate(rows):
        c = DOMCOL[dom]; ax.plot([lo, hi], [i, i], color=c, lw=2.4, solid_capstyle='round'); ax.plot(m, i, 'o', ms=8.5, color=c if pre else 'white', mec=c, mew=1.8, zorder=3)
        ax.text(max(hi, m, 0) + xl * 0.02, i, fmtv(m, lo, hi), va='center', fontsize=8.3)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8); ax.set_xlabel(unit)
forest(axes[0], ROWSA, 'Reduction in missed cases vs the average member (%)', 60, lambda m, lo, hi: f'{m:.0f}% ({lo:.0f} to {hi:.0f})'); axes[0].set_xlim(-2, 62)
panel(axes[0], 'a', 'Against the average member (relative)', 'Filled: pre-specified; open: secondary')
forest(axes[1], ROWSB, 'Missed cases avoided vs the better member, per 10,000 images', 60, lambda m, lo, hi: f'{m:.1f} ({lo:.1f} to {hi:.1f})'); axes[1].set_xlim(-28, 52)
panel(axes[1], 'b', 'Against the better member, chosen with hindsight (count scale)', 'Filled: pre-specified primary; open: secondary; below 0, the pair misses more')
ax = axes[2]
SP = [('HIBA (Argentina)\nAny skin cancer', (17, 12, 22), (27, 18, 37)), ('HIBA (Argentina)\nMelanoma', (7, 2, 12), (26, 14, 40)), ('PAD-UFES-20 (Brazil)\nAny skin cancer', (10, 7, 12), (20, 14, 26))]
for i, (lab, avg, best) in enumerate(SP):
    for (m, lo, hi), dy, mk, c in ((avg, -0.17, 'o', GREY), (best, 0.17, 'D', VERM)):
        ax.plot([lo, hi], [i + dy] * 2, color=c, lw=2.4, solid_capstyle='round'); ax.plot(m, i + dy, mk, ms=8, color=c, mec='white', mew=0.8, zorder=3)
        if mk == 'D': ax.text(hi + 1.2, i + dy, f'{m:.0f}% ({lo:.0f} to {hi:.0f})', va='center', fontsize=8.3, color=VERM)
ax.set_yticks(range(len(SP))); ax.set_yticklabels([s[0] for s in SP]); ax.invert_yaxis(); ax.axvline(0, color='#333333', lw=0.8); ax.set_xlim(-2, 55)
ax.set_xlabel('Reduction in missed cases vs the average member (%)')
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=GREY, ms=7, label='average over all pairs'), Line2D([], [], marker='D', ls='', color=VERM, ms=7, label='strongest partner (own validation score)')],
          fontsize=7.6, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=2)
panel(ax, 'c', 'Exploratory: the strongest partner (skin)')
title(fig, 'Figure 2. A second reader at the same referral volume, against the average member and against the better member (95% intervals resample patients or image groups)')
fig.tight_layout(); save(fig, 'Fig2_benefit')

saved Fig2_benefit


In [4]:
# Figure 7: an exploratory decision framework, with regret panels from Part A (matched evaluation cohorts)
K7 = '100'
def summ(k, excl=('DDR',)):   # population-balanced mean and largest roster mean over the non-DDR rosters available at every pilot size
    keys = [r for r in RESA if all(kk in RESA[r] for kk in ('25', '50', '100')) and r.split('_')[0] not in excl]; pops = sorted({r.split('_')[0] for r in keys}); out = {}
    for p in [q for q in POL if q != 'better']:
        v = {r: RESA[r][k][p][0] for r in keys}
        out[p] = (float(np.mean([np.mean([x for r, x in v.items() if r.split('_')[0] == pp]) for pp in pops])), float(max(v.values())))
    return out
SX = summ(K7)
STR7 = [('blind', 'Pick one blind (same images)', GREY, 'o'), ('pick_auc', 'Always use the pilot winner', ORANGE, 's'), ('pair_max', 'Always pair', BLUE, 'D'), ('adapt_auc_max', 'Pilot test, then pick or pair', GREEN, '^')]
f1 = lambda x: f'{np.floor(10 * x + 0.5) / 10:.1f}'; pb = lambda p: SX[p][0]; lw_ = lambda p: SX[p][1]
rank_fl = [DET[s]['rank_corr']['siblings_flagged'] for s in SITES5]; sib_bm = BRR['B4_sibling_test']['repeat_runs_flagged']
fig = plt.figure(figsize=(17.5, 10.4))
ax = fig.add_axes([0.0, 0.03, 0.57, 0.9]); blank(ax)
ax.text(0.81, 0.985, 'Evidence in this study', ha='center', fontsize=10.5, weight='bold', color='#666666', transform=ax.transAxes)
STEPS = [('1  Judge a second system at the referral volume you can absorb',
          'A pair refers the same number of images as one system. Referring\nevery case either system flags catches more, but only by referring more.',
          'Same volume: fewer misses than the average member\nin all five populations; against the better member,\nthe result depended on population and outcome.', BLUE, '#EAF2FA'),
         ('2  Weigh partner strength as well as diversity',
          'Different training data means fewer shared misses, but a weaker\npartner can make a strong system worse.',
          'Fewer shared misses with different data (pre-specified,\nboth domains). Added catch at equal strength is unresolved;\nin BRSET, a weaker DDR-trained partner left EyePACS-\ntrained systems 10.6 more misses per 10,000.', BLUE, '#EAF2FA'),
         ('3  Test the candidates on local cases, then pick or pair',
          'Draw about 100 severe cases and 100 others from many patients.\nIf one system is clearly stronger (patient-clustered interval\nfor the AUC difference), use it alone; otherwise run both.',
          f'Population-balanced mean regret per 10,000 at 100 + 100,\nexcluding DDR (too few pilot groups), and largest roster\nmean: pilot test {f1(pb("adapt_auc_max"))} ({f1(lw_("adapt_auc_max"))}); pair {f1(pb("pair_max"))} ({f1(lw_("pair_max"))}); pick {f1(pb("pick_auc"))} ({f1(lw_("pick_auc"))}).\nDeveloped on these data; needs independent validation.', GREEN, '#E7F4EF'),
         ('4  Check for close relatives on unlabelled local images',
          "Compare how the candidates rank the programme's own images;\nflag agreement above that among systems known to differ.",
          f'Pre-specified kappa rule flagged {p0(sib_bm)} of BRSET siblings\n(target 80%); score ranks flagged {p0(min(rank_fl))} to {p0(max(rank_fl))}\nacross populations (exploratory, revised after the test).', BLUE, '#EAF2FA'),
         ('5  Choose the referral share for local capacity and prevalence',
          'The share used here kept referrals in proportion to local\nreferable disease; other shares were not validated.',
          'Neither pre-specified test found the predicted reversal\n(DDR at a copied 25% share; EyePACS enriched to 40.6%);\nbenefit held across shares (Supplementary Figs 1 and 2).', BLUE, '#EAF2FA')]
y0, hh, gap = 0.80, 0.135, 0.045
for i, (h, b, ev, ec, fc) in enumerate(STEPS):
    y = y0 - i * (hh + gap)
    box(ax, 0.0, y, 0.6, hh, h, b, ec, fc, hs=10, bs=8.6, lw=1.6 if ec == GREEN else 1.4)
    box(ax, 0.63, y + 0.008, 0.365, hh - 0.016, None, ev, '#BDBDBD', '#F7F7F7', bs=7.7, lw=1.0, ital=True)
    ax.plot([0.6, 0.63], [y + hh / 2] * 2, color='#BDBDBD', lw=1, transform=ax.transAxes)
    if i < len(STEPS) - 1: arrow(ax, 0.3, y, 0.3, y - gap + 0.002, col=BLUE)
ax.text(0.0, y - 0.035, 'Expect the most benefit for conditions each system rarely misses; where all systems miss the same cases\n(melanoma among moles), plan another kind of second look.',
        fontsize=8.4, color='#444444', va='top', transform=ax.transAxes)
ax.text(0.0, 1.0, 'a  An exploratory framework for choosing and checking a second AI reader (not a validated protocol)', fontsize=10.5, weight='bold', va='bottom', transform=ax.transAxes)
gs = fig.add_gridspec(3, 2, left=0.68, right=0.99, top=0.9, bottom=0.11, hspace=0.75, wspace=0.32)
PTC = {'main roster': (BLUE, 'o', 'pair'), 'both EyePACS-trained': (BLUE, 'o', 'both EyePACS-trained'), 'one of each': (GREEN, '^', 'one of each'), 'both DDR-trained': (VERM, 's', 'both DDR-trained')}
for j, (key, lt, ttl) in enumerate((('EyePACS_severe', 'b', 'Close candidates: EyePACS'), ('BRSET_mixed_severe', 'c', 'Far apart: BRSET, mixed sources'))):
    a_ = fig.add_subplot(gs[0, j]); d = PAIRS[PAIRS.roster == key]; mx = d.gap_per_10k.max() * 1.05
    a_.fill_between([0, mx], [0, mx / 2], [0, mx], color='#DDDDDD', alpha=0.5, lw=0); a_.plot([0, mx], [0, mx / 2], ls='--', color='#777777', lw=1); a_.axhline(0, color='#333333', lw=0.8)
    for t, g_ in d.groupby('pair_type'):
        c, mk, lab = PTC[t]; a_.scatter(g_.gap_per_10k, g_.pair_regret_per_10k, s=9, alpha=0.5, color=c, marker=mk, lw=0, label=lab, rasterized=True)
    a_.set_xlabel('Gap between the two systems\n(missed per 10,000)', fontsize=7.8); a_.set_ylabel('Pair regret (missed per 10,000)' if j == 0 else '', fontsize=7.8)
    a_.set_title(f'{lt}  {ttl}', loc='left', fontsize=9, weight='bold')
    if j == 0: a_.legend(handles=[Line2D([], [], ls='--', color='#777777', label='blind pick, expected'), Patch(color='#DDDDDD', label='up to the worst-case blind pick'),
                                  Line2D([], [], ls='', marker='o', color=BLUE, ms=4, label='pair (below 0: beats the better system)')], fontsize=6.4, loc='upper left', handletextpad=0.3)
    if j == 1: a_.legend(fontsize=6.6, loc='upper left', markerscale=1.6, handletextpad=0.2)
for row, (lt, keys, ttl) in enumerate((('d', ['EyePACS_severe', 'BRSET_severe', 'BRSET_mixed_severe'], 'Retina: regret by strategy, pilot 100 + 100, same evaluation images (DDR: Suppl. Fig. 11)'),
                                      ('e', ['HIBA_mel', 'HIBA_mal', 'PAD_mal'], 'Skin: regret by strategy, pilot 100 + 100, same evaluation images')), start=1):
    a_ = fig.add_subplot(gs[row, :]); a_.axvline(0, color='#333333', lw=0.8); keys = [k for k in keys if K7 in RESA.get(k, {})]
    for i, k in enumerate(keys):
        for (p, lab, c, mk), dy in zip(STR7, (-0.27, -0.09, 0.09, 0.27)):
            m, lo, hi = RESA[k][K7][p]; a_.plot([lo, hi], [i + dy] * 2, color=c, lw=1.2); a_.scatter(m, i + dy, color=c, marker=mk, s=30, edgecolor='white', lw=0.5, zorder=3, label=lab if i == 0 else None)
    a_.set_yticks(range(len(keys))); a_.set_yticklabels([LBL[k].replace(', ', '\n', 1) for k in keys], fontsize=7.6); a_.invert_yaxis(); a_.tick_params(axis='y', length=0)
    a_.set_xlabel('Regret: missed per 10,000 above the better system (0 = knowing which is stronger)\n95% intervals resample evaluation patients, conditional on the pilot draws', fontsize=7.6)
    a_.set_title(f'{lt}  {ttl}', loc='left', fontsize=9, weight='bold')
    if row == 2: a_.legend(fontsize=7.2, loc='upper center', bbox_to_anchor=(0.45, -0.34), ncol=4, handletextpad=0.2, columnspacing=1.0)
title(fig, 'Figure 7. Choosing and checking a second AI reader: an exploratory framework')
save(fig, 'Fig7_decision_flow')

saved Fig7_decision_flow


In [5]:
# Supplementary Figure 11: the pilot rule with patient-clustered inference (Part A)
KS = [k for k in ('25', '50', '100') if k in SUMA]; SXK = {k: summ(k) for k in KS}
SN = [('pick_auc', 'Always use the pilot winner', ORANGE, 's', '-'), ('pair_max', 'Always pair (maximum rank)', BLUE, 'D', '-'), ('pair_mean', 'Always pair (mean rank)', SKY, 'D', '--'),
      ('adapt_auc_max', 'Pilot test (clustered AUC), then pick or pair', GREEN, '^', '-'), ('adapt_vol_max', 'Endpoint-aligned pilot test, then pick or pair', PINK, 'v', '-'), ('blind', 'Pick one blind', GREY, 'o', ':')]
fig, axes = plt.subplots(1, 3, figsize=(17, 5.4), gridspec_kw={'width_ratios': [1, 1, 1.25], 'wspace': 0.45})
for a_, fld, lt, ttl in ((axes[0], 'population_balanced_mean', 'a', 'Population-balanced mean regret, same six rosters'), (axes[1], 'largest_observed_roster_mean', 'b', 'Largest observed roster mean, same six rosters')):
    for s, lab, c, mk, ls in SN:
        v = [SXK[k][s][0 if fld == 'population_balanced_mean' else 1] for k in KS]
        a_.plot([int(k) for k in KS], v, marker=mk, color=c, ls=ls, lw=1.8, ms=6, mec='white', label=lab)
    a_.axhline(0, color='#333333', lw=0.8); a_.set_xticks([int(k) for k in KS]); a_.set_xticklabels([f'{k} + {k}' for k in KS])
    a_.set_xlabel('Pilot size (severe cases + others)'); a_.set_ylabel('Regret (missed per 10,000)'); panel(a_, lt, ttl)
fig.legend(*axes[0].get_legend_handles_labels(), fontsize=7.6, loc='lower left', bbox_to_anchor=(0.02, -0.06), ncol=3)
a_ = axes[2]; keys = [kk for kk in RESA if '100' in RESA[kk]]
for i, kk in enumerate(keys):
    for (d, c, dy, lab) in (('adapt_auc_max_minus_pair_max', GREEN, -0.15, 'pilot test minus always pair'), ('adapt_auc_max_minus_pick_auc', ORANGE, 0.15, 'pilot test minus always pick')):
        m, lo, hi = RESA[kk]['100']['diff'][d]; a_.plot([lo, hi], [i + dy] * 2, color=c, lw=1.6); a_.plot(m, i + dy, 'o', color=c, ms=5, label=lab if i == 0 else None)
a_.axvline(0, color='#333333', lw=0.8); a_.set_yticks(range(len(keys))); a_.set_yticklabels([LBL[k] + (' (sparse groups)' if k.startswith('DDR') else '') for k in keys], fontsize=7.6); a_.invert_yaxis()
a_.set_xlabel('Difference in regret per 10,000 (below 0: the pilot test does better)\n95% intervals resample evaluation patients, conditional on the pilot draws'); a_.legend(fontsize=7.4, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2); panel(a_, 'c', 'Paired differences at 100 + 100, same evaluation images')
title(fig, 'Supplementary Figure 11. A local pilot, then pick or pair, with patient-clustered inference (exploratory; scored on patients outside the pilot; 100 draws; a, b: the six non-DDR rosters available at every pilot size)')
fig.subplots_adjust(left=0.05, right=0.99, bottom=0.2, top=0.86); save(fig, 'SuppFig11_pilot_then_pick_or_pair')
drive.flush_and_unmount(); print('done')

saved SuppFig11_pilot_then_pick_or_pair


ValueError: flush_and_unmount failed